In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Train basic gameplay model
# Stage 1 - SelfplayEnv with Random opponent

from DeepLearning.PPO import MaskablePPO
from DeepLearning.Thesis.Environments.Setup import SetupAgentCities
from DeepLearning.Thesis.DeepLearning.Thesis.Environments.SelfPlay import SelfPlayBase
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.Thesis.Setup.getActionMaskSetup import getSetupActionMask
import os
from DeepLearning.PPO import MaskablePPO
from DeepLearning.Thesis.Observations.get_observation_full import getObservationFull, lowerBound, upperBound
#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayBase()
env.selfPlay = False # for random opponents
actionMask = getActionMask
observation = getObservationFull


os.environ["UPDATE_MODELS_DIST"] = "False"
netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_FullObservation_GameplayStage_5M_SelfPlay"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

# model = MaskablePPO.load("DeepLearning/Thesis/Setup/Models/SetupRandom/model_332400_5.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=5_000_000, tb_log_name=saveName, reset_num_timesteps=False)

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute '_function_setstate' on <module 'cloudpickle.cloudpickle' from 'D:\\ProgramData\\Anaconda3_\\envs\\catan\\Lib\\site-packages\\cloudpickle\\cloudpickle.py'>
  warnings.warn(
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute '_function_setstate' on <module 'cloudpickle.cloudpickle' from 'D:\\ProgramData\\Anaconda3_\\envs\\catan\\Lib\\site-packages\\cloudpickle\\cloudpickle.py'>
  warnings.warn(


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Policy device: cuda:0
Logging to ./tensorboard_logs_thesis/ZKA_FullObservation_GameplayStage_5M_SelfPlay_0


D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\utils.py:168: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\utils.py:214: UserWarning: constant_fn() is deprecated, please use ConstantSchedule() instead
  warnings.warn("constant_fn() is deprecated, please use ConstantSchedule() instead")
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute '_function_setstate' on <module 'cloudpickle.cloudpickle' from 'D:\\ProgramData\\Anaconda3_\\envs\\catan\\Lib\\site-packages\\cloudpickle\\cloudpickle.py'>
  warnings.warn(
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\st

CheckingWinRate(Distribution): 0.03
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 101      |
|    ep_rew_mean     | 24       |
| time/              |          |
|    fps             | 206      |
|    iterations      | 1        |
|    time_elapsed    | 9        |
|    total_timesteps | 2048     |
---------------------------------
CheckingWinRate(Distribution): 0.09
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90.3        |
|    ep_rew_mean          | 31.6        |
| time/                   |             |
|    fps                  | 191         |
|    iterations           | 2           |
|    time_elapsed         | 21          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.015408615 |
|    clip_fraction        | 0.196       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.89       |


KeyboardInterrupt: 

In [ ]:
print(model.observation_space.shape)
print(env.observation_space.shape)

In [ ]:
model.save(savePath)
print(savePath)

In [ ]:
 # Stage 2 - TurnLimitDense + AgainstRandom

from DeepLearning.Thesis.DeepLearning.Thesis.Environments.TurnLimitDense import TurnLimitDense
from Agents.AgentRandom2 import AgentRandom2
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.Thesis.Observations.get_observation_full import getObservationFull
from DeepLearning.PPO import MaskablePPO
import os
os.environ["TURN_LIMIT"] = "30"
os.environ["UPDATE_MODELS_UNIFORM"] = "False"
os.environ["UPDATE_MODELS_DIST"] = "False"

#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = TurnLimitDense(players=[
    AgentRandom2("P0", 0),
    AgentRandom2("P1", 1),
    AgentRandom2("P2", 2),
    AgentRandom2("P3", 3)
])
actionMask = getActionMask
observation = getObservationFull

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_Try2_GameplayStage_+3M_Turnlimit"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

model = MaskablePPO.load("DeepLearning/Models/ZKA_model/model_2330624_177.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [ ]:
model.save(savePath)
print(savePath)

In [2]:
 # Stage 3 - Selfplay
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


from DeepLearning.Thesis.DeepLearning.Thesis.Environments.SelfPlay import SelfPlayZKA
from Agents.AgentModel import AgentModel
from Agents.AgentRandom2 import AgentRandom2
from DeepLearning.GetActionMask import getActionMask, getActionMaskTrading
from DeepLearning.Thesis.Observations.get_observation_full import getObservationFull, lowerBound, upperBound
from DeepLearning.PPO import MaskablePPO
import os



#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayZKA(selfPlay = True)
env.selfPlay = True
env.denseRewards = False
env.bankTradeRewards = False
actionMask = getActionMask
observation = getObservationFull
os.environ["UPDATE_MODELS_DIST"] = "False"

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_FullObservation_GameplayStage_Selfplay"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")f

#model = MaskablePPO.load("DeepLearning/Models/ZKA_model/Better_Densereward.zip", env=env)
print(model.observation_space.shape)
print(env.observation_space.shape)

model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=6_000_000, tb_log_name=saveName, reset_num_timesteps=False)

CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU
(1233,)
(1233,)
Policy device: cuda:0
Logging to ./tensorboard_logs_thesis/ZKA_FullObservation_GameplayStage_Selfplay_0
CheckingWinRate(Distribution): 0.9
Updating opponents to: model_2409978_41, model_2338816_85.zip, model_2371584_92.zip
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 55.8     |
|    ep_rew_mean     | 83       |
| time/              |          |
|    fps             | 264      |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 2409978  |
---------------------------------
Using opponents: model_2409978_41, model_2338816_85.zip, model_2371584_92.zip
CheckingWinRate(Distribution): 0.35
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 55          |
|    ep_rew_mean          | 87.1        |
| time/                   |             |
|    fps                  | 

KeyboardInterrupt: 

In [3]:
model.save(savePath)
print(savePath)

DeepLearning/Models/ZKA_model/ZKA_FullObservation_GameplayStage_Selfplay
